# Stability vs. Flexibility — Coding-Assignment Sandbox (A1–A6)

A hands-on, **fill-in-the-blank** playground for the coding assignments in
[`docs/stability_flexibility_coding_assignments.md`](../../../docs/stability_flexibility_coding_assignments.md):

| # | Assignment | Skeleton | You build |
|---|---|---|---|
| **A1** | Per-electrode two-way **ANOVA** electrode definition | `a1_anova_labels.py` | `_anova_interaction_stats`, `per_electrode_anova_labels` |
| **A2** | Conjunction **permutation null** + **threshold sweep** | `a2_conjunction_null_sweep.py` | `conjunction_permutation_null`, `conjunction_threshold_sweep` |
| **A3** | **Anatomy**: ROI join + coverage-conditioned enrichment | `a3_anatomy.py` | `attach_roi`, `build_coverage_matrix`, `roi_group_enrichment_test` |
| **A4** | **Cross-decoding**: pseudo-trials + label transfer + mean-removal control | `a4_cross_decoding.py` | `build_pseudo_trials`, `remove_condition_means`, `cross_decode` |
| **A5** | **Timing**: interaction time course, 50%-of-peak onset, jackknife | `a5_stability_flexibility_timing.py` | `interaction_time_course`, `onset_50pct_peak`, `peak_latency`, `jackknife_onset_difference` |
| **A6** | **Brain–behavior** correlation (across- and within-subject) | `a6_brain_behavior.py` | `subject_level_brain_behavior`, `trialwise_brain_behavior` |

Each drops into its target module (see the assignments doc). **A1–A2** are the deepest,
end-to-end sections; **A3–A6** cover the rest of the battery — some pieces that fundamentally
need real data or heavy deps (brain-surface rendering, the production `Decoder`) are shown as
reference + a runnable compact stand-in, and clearly flagged.

### How this notebook works
1. **Setup** loads real iEEG data for **1–2 subjects, clipped to LPFC electrodes** — and
   automatically **falls back to a synthetic ground-truth dataset** if the real data isn't
   reachable (e.g. you're off the cluster / Box). Everything downstream runs either way.
2. Each **Task** gives you a stub cell that raises `NotImplementedError`. Fill it in.
3. Stuck? **Reveal hints on demand** — no scrolling past spoilers:
   - `reveal("a1_stats_hint1")` … tiered nudges: *what to reach for* (libraries/functions).
   - `reveal("a1_stats_solution")` … the reference implementation, to read/copy.
   - `load_reference("a1_stats")` … inject the reference into the namespace so you can keep
     going even if you'd rather not implement that piece yet.
4. Each task ends with an **acceptance-check** cell that encodes the assignment's success
   criteria (agreement with the nonparametric labels, near-null cross-controls, invariant
   permutations, OR≈1 on independent data).

> The `reveal(...)` and `load_reference(...)` helpers are defined in the setup section.
> Hints/solutions live in a registry so they stay out of sight until you ask.

---
## 0 · Setup — imports, paths, reveal helpers

In [ ]:
%matplotlib inline
# --- path setup so `src...` and `dcc_scripts...` import cleanly ---------------
import os, sys, textwrap, warnings
warnings.simplefilter("ignore")

# Walk up to the repo root (the folder containing `src/` and `docs/`).
_here = os.getcwd()
_root = _here
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "src")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("repo root:", _root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# statsmodels pieces you'll use in A1 / A6
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests

# scipy stats used in A3 / A5 / A6
from scipy.stats import chi2_contingency, pearsonr
from scipy.stats import t as _tdist

# A4 uses a compact classifier (the production drop-in wraps the project Decoder)
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    _HAS_SKLEARN = True
except Exception:
    _HAS_SKLEARN = False
    print("(sklearn not installed — A4 cells will be skipped; `pip install scikit-learn`)")

# helpers already in the segregation module — reuse these, don't rebuild them
from src.analysis.stats.stability_flexibility_segregation import (
    _canonical_labels, _interaction_effect, _is_interaction,
    resolve_contrasts, finalize_contrasts,
    per_electrode_labels, cmh_conjunction, subject_clustered_corr,
    compute_sensitivities, add_responsiveness, prepare_continuous,
    _synthetic_df,
)
print("segregation module imported OK")

### The reveal registry
Run this once. It only *stores* hints and solutions — nothing is shown until you call
`reveal(...)`. (Peeking here is allowed, but it's the whole answer key.)

In [ ]:

from IPython.display import display, Markdown, Code

_REVEALS = {}   # key -> ("md" | "code", text)

def _register(key, kind, text):
    _REVEALS[key] = (kind, textwrap.dedent(text).strip("\n"))

def reveal(key):
    """Show a hint (markdown) or a reference solution (code) on demand."""
    if key not in _REVEALS:
        print(f"no reveal named {key!r}. Available:")
        for k in sorted(_REVEALS): print("   ", k)
        return
    kind, text = _REVEALS[key]
    if kind == "code":
        display(Markdown(f"**Reference solution — `{key}`**  "
                         f"(read/copy above, or run `load_reference('{key}')`):"))
        display(Code(text, language="python"))
    else:
        display(Markdown(text))

def load_reference(key):
    """Exec a reference solution into the notebook namespace so you can proceed.
    Accepts the short key ('a1_stats') or the full one ('a1_stats_solution')."""
    resolved = key if key in _REVEALS else f"{key}_solution"
    kind, text = _REVEALS.get(resolved, (None, None))
    if kind != "code":
        raise KeyError(f"{key!r} is not a loadable code reference "
                       f"(try one of: {[k for k in _REVEALS if k.endswith('_solution')]})")
    exec(compile(text, f"<reference:{resolved}>", "exec"), globals())
    print(f"✓ loaded reference implementation: {resolved}")

def list_reveals(prefix=""):
    for k in sorted(_REVEALS):
        if k.startswith(prefix):
            print(f"  {k:28s} [{_REVEALS[k][0]}]")


# ------------------------------- A1 hints/solutions -------------------------
_register("a1_stats_hint1", "md", """
**A1 · `_anova_interaction_stats` — hint 1 (the trap).**
This is a per-electrode *two-way* ANOVA, but the point of the assignment is a
**leakage trap**: the proportion cells are deliberately unequal (~75/25). With
statsmodels' default **treatment** coding, the Type III interaction sum-of-squares is
*not* orthogonal to the main effects, so a pure congruency/switch **main effect leaks
into the "interaction"**. Fix it with **Type III SS + sum/effect coding** — the
parametric analogue of the equal-cell-weight difference-of-differences the module
already uses in `_interaction_effect`.
""")

_register("a1_stats_hint2", "md", """
**A1 · `_anova_interaction_stats` — hint 2 (what to reach for).**
- `statsmodels.formula.api.ols` for the fit; `statsmodels.stats.anova.anova_lm(model, typ=3)`.
- Sum-code **both** factors in the formula so Type III is orthogonal:
  `f"{hg_col} ~ C({cond_col}, Sum) * C({mod_col}, Sum)"`.
- The interaction row's index is literally `f"C({cond_col}, Sum):C({mod_col}, Sum)"`;
  read `'F'` and `'PR(>F)'` from it.
- Wrap the fit in `try/except` and return `{'F': np.nan, 'p': np.nan}` on a singular
  design (an electrode missing a cell) — mirror how the module's effect helpers return
  NaN for too-few trials.
""")

_register("a1_stats_solution", "code", """
def _anova_interaction_stats(elec_df, cond_col, mod_col, hg_col="hg"):
    \"\"\"One electrode's two-way Type III ANOVA; returns the interaction F and p.\"\"\"
    try:
        formula = f"{hg_col} ~ C({cond_col}, Sum) * C({mod_col}, Sum)"
        model = smf.ols(formula, data=elec_df).fit()
        aov = anova_lm(model, typ=3)                       # Type III
        inter = f"C({cond_col}, Sum):C({mod_col}, Sum)"
        return {"F": float(aov.loc[inter, "F"]),
                "p": float(aov.loc[inter, "PR(>F)"])}
    except Exception:
        return {"F": np.nan, "p": np.nan}                  # singular / missing cell
""")

_register("a1_labels_hint1", "md", """
**A1 · `per_electrode_anova_labels` — hint 1 (scaffold + sign).**
- Canonicalize once up front so the `_scond/_smod/_fcond/_fmod` cell labels exist:
  `contrasts = finalize_contrasts(df, resolve_contrasts(contrast_mode, contrasts))`
  then `work = _canonical_labels(df, contrasts)`.
- Loop `work.groupby(["subject", "electrode"])`; call `_anova_interaction_stats`
  twice: stability = congruency × incongruent_proportion, flexibility =
  switchType × switch_proportion.
- The F-test is **unsigned**. Get the sign from the module's *own* estimator so it
  matches the §2 correlation exactly:
  `np.sign(_interaction_effect(hg, cond, mod, "cohens_d", alpha))` using
  `_scond/_smod` (stability) and `_fcond/_fmod` (flexibility).
""")

_register("a1_labels_hint2", "md", """
**A1 · `per_electrode_anova_labels` — hint 2 (FDR, flags, cross-controls).**
- FDR across electrodes: `q = multipletests(p.fillna(1), method="fdr_bh")[1]` for
  `p_cong` and `p_switch` separately.
- Flags: `S = (q_cong < alpha)`, `F = (q_switch < alpha)`. You *may* additionally
  require the sign to be positive (predicted direction) — decide & document. This
  reference exposes it via `require_sign` (default `False`, to match the
  nonparametric sibling `per_electrode_labels`).
- **Cross controls (report only, never select on):** congruency × switch_proportion
  and switchType × incongruent_proportion. On the synthetic generator there is no
  true cross-effect, so their p-values should be ~uniform.
""")

_register("a1_labels_solution", "code", """
def per_electrode_anova_labels(df, alpha=0.05, contrast_mode="proportion",
                               contrasts=None, include_cross_controls=True,
                               require_sign=False):
    \"\"\"Parametric per-electrode S/F labels from the two-way interaction ANOVA.
    Drop-in `labels` for cmh_conjunction (columns match per_electrode_labels).\"\"\"
    contrasts = finalize_contrasts(df, resolve_contrasts(contrast_mode, contrasts))
    work = _canonical_labels(df, contrasts)               # attaches _scond/_smod/...
    recs = []
    for (subj, elec), g in work.groupby(["subject", "electrode"]):
        st = _anova_interaction_stats(g, "congruency", "incongruent_proportion")
        fl = _anova_interaction_stats(g, "switchType", "switch_proportion")
        s_sign = np.sign(_interaction_effect(g["hg"].to_numpy(),
                                             g["_scond"].to_numpy(),
                                             g["_smod"].to_numpy(), "cohens_d", alpha))
        f_sign = np.sign(_interaction_effect(g["hg"].to_numpy(),
                                             g["_fcond"].to_numpy(),
                                             g["_fmod"].to_numpy(), "cohens_d", alpha))
        rec = dict(subject=subj, electrode=elec,
                   p_cong=st["p"], F_cong=st["F"], s_sign=s_sign,
                   p_switch=fl["p"], F_switch=fl["F"], f_sign=f_sign)
        if include_cross_controls:                        # specificity controls
            rec["p_cross_cs"] = _anova_interaction_stats(
                g, "congruency", "switch_proportion")["p"]
            rec["p_cross_si"] = _anova_interaction_stats(
                g, "switchType", "incongruent_proportion")["p"]
        recs.append(rec)
    out = pd.DataFrame(recs)
    out["q_cong"]   = multipletests(out["p_cong"].fillna(1),   method="fdr_bh")[1]
    out["q_switch"] = multipletests(out["p_switch"].fillna(1), method="fdr_bh")[1]
    S = out["q_cong"] < alpha
    F = out["q_switch"] < alpha
    if require_sign:
        S = S & (out["s_sign"] > 0)
        F = F & (out["f_sign"] > 0)
    out["S"] = S.astype(int)
    out["F"] = F.astype(int)
    return out
""")

# ------------------------------- A2 hints/solutions -------------------------
_register("a2_null_hint1", "md", """
**A2 · `conjunction_permutation_null` — hint 1 (what to shuffle).**
Shuffle **F within each subject** (leave S untouched). That holds every subject's
S-count and F-count fixed and randomizes only the *pairing* — the exact null CMH
assumes, and it mirrors the within-subject permutation in `subject_clustered_corr`.
Global shuffling would break the subject nesting and manufacture significance.
`observed = int(((S==1) & (F==1)).sum())`.
""")

_register("a2_null_hint2", "md", """
**A2 · `conjunction_permutation_null` — hint 2 (mechanics + the invariant).**
- Precompute per-subject row-index groups (`[np.where(subj==s)[0] for s in unique]`)
  and, each permutation, set `Fp[idx] = F[rng.permutation(idx)]`.
- Two-sided p vs the null **mean**: `(sum(|null-mean| >= |obs-mean|) + 1)/(n_perm+1)`.
- **Acceptance invariant:** every permutation must leave each subject's `S.sum()` and
  `F.sum()` unchanged — only the pairing moves. The acceptance cell asserts this.
""")

_register("a2_null_solution", "code", """
def conjunction_permutation_null(labels, n_perm=10000, seed=0):
    \"\"\"Within-subject permutation null for the count of 'both' (S==1 & F==1) electrodes.\"\"\"
    lab = labels.dropna(subset=["S", "F"]).copy()
    S = lab["S"].to_numpy().astype(int)
    F = lab["F"].to_numpy().astype(int)
    subj = lab["subject"].to_numpy()
    groups = [np.where(subj == s)[0] for s in np.unique(subj)]
    observed = int(((S == 1) & (F == 1)).sum())
    rng = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        Fp = F.copy()
        for idx in groups:                 # shuffle F WITHIN each subject only
            Fp[idx] = F[rng.permutation(idx)]
        null[i] = int(((S == 1) & (Fp == 1)).sum())
    mean = null.mean()
    p = (np.sum(np.abs(null - mean) >= abs(observed - mean)) + 1) / (n_perm + 1)
    z = (observed - mean) / null.std() if null.std() > 0 else np.nan
    return dict(observed=observed, null=null, p_two_sided=float(p), z=float(z))
""")

_register("a2_sweep_hint1", "md", """
**A2 · `conjunction_threshold_sweep` — hint 1 (reuse, don't rebuild).**
The sweep is a *loop*: for each threshold, get a fresh labels table from the
`labels_by_threshold` callable, then call the existing `cmh_conjunction(labels)`.
Record `n_S`, `n_F`, `n_both`, `mh_odds_ratio`, and `res['cmh'].pvalue`. Keeping the
callable makes the function agnostic to whether you threshold on q-values (ANOVA /
permutation) or on effect-size percentiles.
""")

_register("a2_sweep_solution", "code", """
def conjunction_threshold_sweep(labels_by_threshold, thresholds):
    \"\"\"Recompute overlap OR + counts across selection thresholds -> tidy DataFrame.\"\"\"
    rows = []
    for t in thresholds:
        lab = labels_by_threshold(t)
        res = cmh_conjunction(lab)
        rows.append(dict(threshold=t,
                         n_S=int(lab.S.sum()), n_F=int(lab.F.sum()),
                         n_both=int(((lab.S == 1) & (lab.F == 1)).sum()),
                         mh_odds_ratio=res["mh_odds_ratio"],
                         cmh_p=res["cmh"].pvalue))
    return pd.DataFrame(rows)
""")

print("reveal registry loaded:")
list_reveals()


# ------------------------------- A3 hints/solutions -------------------------
_register("a3_hint1", "md", """
**A3 — hint 1 (the trap: coverage bias).** iEEG coverage is *clinical*, so a raw ROI
difference between selectivity groups can just reflect **where electrodes happen to be**.
Every anatomical claim must be conditioned on coverage: restrict to ROIs sampled in ≥ *k*
subjects, report per-ROI coverage, and build the null by permuting the group label
**within subject** (so it respects nesting *and* coverage simultaneously).
""")

_register("a3_hint2", "md", """
**A3 — hint 2 (what to reach for).**
- `attach_roi`: `labels["electrode"].map(electrodes_to_rois)`; derive `group` from (S, F).
- `build_coverage_matrix`: `pivot_table(index="subject", columns="roi", aggfunc="max",
  fill_value=0).astype(bool)`.
- `roi_group_enrichment_test`: `scipy.stats.chi2_contingency(table)[0]` as the statistic;
  keep ROIs with `coverage.sum(0) >= min_subjects`; permutation p by shuffling `group`
  within each subject's row indices (`rng.permutation(idx)`), rebuilding the group×ROI
  contingency each time. Reindex the permuted table to fixed rows/cols so the χ² is comparable.
""")

_register("a3_solution", "code", """
def attach_roi(labels, electrodes_to_rois):
    \"\"\"Add 'roi' (electrode->region) and a 4-way 'group' from (S, F).\"\"\"
    out = labels.copy()
    out["roi"] = out["electrode"].map(electrodes_to_rois)
    def grp(s, f):
        if s == 1 and f == 1: return "both"
        if s == 1: return "S_only"
        if f == 1: return "F_only"
        return "neither"
    out["group"] = [grp(s, f) for s, f in zip(out["S"], out["F"])]
    return out

def build_coverage_matrix(labels_with_roi):
    \"\"\"subject x ROI boolean: does subject have ANY electrode in that ROI?\"\"\"
    d = labels_with_roi.dropna(subset=["roi"]).assign(_x=1)
    return d.pivot_table(index="subject", columns="roi", values="_x",
                         aggfunc="max", fill_value=0).astype(bool)

def _contingency(gp, roi, groups, rois):
    tab = pd.crosstab(pd.Series(list(gp), name="group"),
                      pd.Series(list(roi), name="roi"))
    return tab.reindex(index=groups, columns=rois, fill_value=0)

def roi_group_enrichment_test(labels_with_roi, coverage, min_subjects=3,
                              n_perm=5000, seed=0,
                              groups_keep=("S_only", "F_only", "both")):
    \"\"\"Coverage-conditioned test: is group membership associated with ROI?\"\"\"
    roi_cov = coverage.sum(axis=0)
    keep = sorted(roi_cov[roi_cov >= min_subjects].index.tolist())   # coverage condition
    d = labels_with_roi.dropna(subset=["roi"])
    d = d[d["roi"].isin(keep) & d["group"].isin(groups_keep)]
    groups = list(groups_keep)
    obs_tab = _contingency(d["group"], d["roi"], groups, keep)
    obs = chi2_contingency(obs_tab)[0]
    subj = d["subject"].to_numpy(); grp = d["group"].to_numpy(); roi = d["roi"].to_numpy()
    gidx = [np.where(subj == s)[0] for s in np.unique(subj)]
    rng = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        gp = grp.copy()
        for idx in gidx:                       # permute group WITHIN subject
            gp[idx] = grp[rng.permutation(idx)]
        null[i] = chi2_contingency(_contingency(gp, roi, groups, keep))[0]
    p = (np.sum(null >= obs) + 1) / (n_perm + 1)
    return dict(rois_tested=keep, observed_stat=float(obs), p=float(p),
                contingency=obs_tab, per_roi_coverage=roi_cov[keep])
""")

# ------------------------------- A4 hints/solutions -------------------------
_register("a4_hint1", "md", """
**A4 — hint 1 (the whole point).** Co-localization ≠ shared *code*. Train a decoder on one
contrast and test on another: if the 'both' electrodes share a representational geometry the
transfer works; if they carry orthogonal codes it fails. Two non-negotiable controls:
**disjoint train/test trials** (circularity) and **per-condition mean removal** — a cross-effect
that vanishes after `remove_condition_means` was a univariate offset, not a code.
""")

_register("a4_hint2", "md", """
**A4 — hint 2 (mechanics).** Subjects don't share trials → build a *pseudopopulation*:
features = electrodes, one **pseudo-trial** = for each electrode, the mean of `n_per_cell`
sampled trials of a given class. Split raw trials into disjoint halves first (train pseudo-trials
from half A, test from half B). Route fit/predict through one **backend helper** so you can
swap the compact `LogisticRegression` for the project `Decoder` (PCA+LDA) by flipping a flag —
`Decoder(categories={0:0,1:1}).fit(X, y).predict(X)` takes `(n_trials, n_features)`. Null =
permute the *test* labels.
""")

_register("a4_solution", "code", """
def _fit_predict(Xtr, ytr, Xte, backend=None, explained_variance=0.8, random_state=0):
    \"\"\"Train on (Xtr,ytr) and predict Xte through a pluggable backend.
    backend: 'decoder' (project PCA+LDA Decoder), 'sklearn' (compact), None/'auto' -> A4_BACKEND.\"\"\"
    b = backend or globals().get("A4_BACKEND", "auto")
    use_dec = (b == "decoder") or (b == "auto" and globals().get("_HAS_DECODER", False))
    if use_dec:
        cats = {int(c): int(c) for c in np.unique(ytr)}          # binary {0,1}
        dec = Decoder(categories=cats, explained_variance=explained_variance,
                      n_splits=3, n_repeats=1, random_state=random_state)
        dec.fit(Xtr, ytr)
        return np.asarray(dec.predict(Xte))
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
    clf.fit(Xtr, ytr)
    return clf.predict(Xte)

def build_pseudo_trials(df, label_col, pos, neg, n_pseudo=60, n_per_cell=8, seed=0):
    \"\"\"Pseudopopulation pseudo-trials: features=electrodes, one row = per-electrode
    mean of n_per_cell sampled trials of a class. Returns (X, y).\"\"\"
    rng = np.random.default_rng(seed)
    electrodes = sorted(df["electrode"].unique())
    pools = {(e, c): df[(df.electrode == e) & (df[label_col] == c)]["hg"].to_numpy()
             for e in electrodes for c in (pos, neg)}
    X, y = [], []
    for c, lab in [(pos, 1), (neg, 0)]:
        for _ in range(n_pseudo):
            feat = []
            for e in electrodes:
                pool = pools[(e, c)]
                if len(pool) == 0:
                    feat.append(np.nan); continue
                idx = rng.choice(len(pool), size=min(n_per_cell, len(pool)), replace=False)
                feat.append(pool[idx].mean())
            X.append(feat); y.append(lab)
    X = np.array(X, float); y = np.array(y)
    good = ~np.isnan(X).any(0)               # drop electrodes missing a class
    return X[:, good], y

def remove_condition_means(X, condition_labels):
    \"\"\"Subtract each condition's per-feature mean (the univariate-offset control).\"\"\"
    Xc = X.astype(float).copy()
    for c in np.unique(condition_labels):
        m = condition_labels == c
        Xc[m] -= Xc[m].mean(0, keepdims=True)
    return Xc

def _disjoint_halves(df, seed=0):
    \"\"\"Split each electrode's trials into two disjoint halves (train vs test pool).\"\"\"
    rng = np.random.default_rng(seed)
    a, b = [], []
    for _, g in df.groupby("electrode"):
        idx = g.index.to_numpy().copy(); rng.shuffle(idx)
        cut = len(idx) // 2
        a.append(idx[:cut]); b.append(idx[cut:])
    return df.loc[np.concatenate(a)], df.loc[np.concatenate(b)]

def cross_decode(df, train_spec, test_spec, strip_condition_means=False, backend=None,
                 n_pseudo=80, n_per_cell=8, n_perm=200, seed=0):
    \"\"\"Train on train_spec contrast, TEST on test_spec contrast (disjoint trials).
    spec = (label_col, pos, neg), e.g. ('congruency','i','c'). Returns acc, null, p.\"\"\"
    dfa, dfb = _disjoint_halves(df, seed)                       # circularity guard
    Xtr, ytr = build_pseudo_trials(dfa, *train_spec, n_pseudo=n_pseudo,
                                   n_per_cell=n_per_cell, seed=seed)
    Xte, yte = build_pseudo_trials(dfb, *test_spec, n_pseudo=n_pseudo,
                                   n_per_cell=n_per_cell, seed=seed + 1)
    k = min(Xtr.shape[1], Xte.shape[1]); Xtr, Xte = Xtr[:, :k], Xte[:, :k]
    if strip_condition_means:                                   # univariate-offset control
        Xtr = remove_condition_means(Xtr, ytr); Xte = remove_condition_means(Xte, yte)
    acc = (_fit_predict(Xtr, ytr, Xte, backend) == yte).mean()
    rng = np.random.default_rng(seed + 7)
    null = np.array([(_fit_predict(Xtr, ytr, Xte, backend) == rng.permutation(yte)).mean()
                     for _ in range(n_perm)])
    p = (np.sum(null >= acc) + 1) / (n_perm + 1)
    return dict(accuracy=float(acc), null_mean=float(null.mean()), p=float(p))
""")

# --- A4 within-block baseline (Fig 9) ---------------------------------------
_register("a4_within_hint", "md", """
**A4 within-block (Fig 9) — hint.** Decode the *same* contrast **within each block level**
and compare accuracies — the decoding analog of the univariate LWPC/LWPS effect. Congruency
should decode **better in mostly-incongruent blocks** (`incongruent_proportion` high), switch
better in mostly-switch blocks. It's just `cross_decode(sub, spec, spec)` on each block subset;
match trial counts first. The across-block *difference* is where the neural cross-effect lives.
""")

_register("a4_within_solution", "code", """
def within_block_decoding_baseline(df, contrast_spec, block_col, backend=None,
                                   n_pseudo=60, n_per_cell=3, n_perm=100, seed=0):
    \"\"\"Decode contrast_spec WITHIN each level of block_col; compare accuracies (Fig 9).\"\"\"
    out = {}
    for b in sorted(df[block_col].dropna().unique()):
        sub = df[df[block_col] == b]
        out[b] = cross_decode(sub, contrast_spec, contrast_spec, backend=backend,
                              n_pseudo=n_pseudo, n_per_cell=n_per_cell, n_perm=n_perm, seed=seed)
    levels = sorted(out)
    diff = out[levels[-1]]["accuracy"] - out[levels[0]]["accuracy"]
    return dict(per_block=out, block_levels=levels, acc_difference=float(diff))
""")

# --- A4 temporal generalization (Fig 10) ------------------------------------
_register("a4_tempgen_hint", "md", """
**A4 temporal generalization (Fig 10) — hint.** Train the decoder at time *t*, test at *t′*,
for all (t, t′) → a train×test accuracy matrix. Off-diagonal generalization = a sustained/stable
code; a narrow diagonal = a moving/phasic code. Needs **time-course** pseudo-trials — build an
`(n_pseudo, n_elec, n_bins)` tensor (per-electrode mean of sampled trials, keeping the time
axis), keep train/test disjoint, and loop the same `_fit_predict` over bin pairs (sub-sample
bins with a `stride` to keep it fast).
""")

_register("a4_tempgen_solution", "code", """
def build_pseudo_trials_tc(df, label_col, pos, neg, n_pseudo=40, n_per_cell=8, seed=0, stride=2):
    \"\"\"Time-course pseudo-trials: X (n_pseudo, n_elec, n_binsub), y, kept bin indices.\"\"\"
    rng = np.random.default_rng(seed)
    electrodes = sorted(df["electrode"].unique())
    T = len(np.asarray(df["hg"].iloc[0])); bins = np.arange(0, T, stride)
    pools = {}
    for e in electrodes:
        for c in (pos, neg):
            sel = (df.electrode == e) & (df[label_col] == c)
            pools[(e, c)] = (np.vstack([np.asarray(v, float) for v in df[sel]["hg"].to_numpy()])
                             if sel.any() else None)
    X, y = [], []
    for c, lab in [(pos, 1), (neg, 0)]:
        for _ in range(n_pseudo):
            rows = []
            for e in electrodes:
                pool = pools[(e, c)]
                if pool is None:
                    rows.append(np.full(len(bins), np.nan)); continue
                idx = rng.choice(len(pool), size=min(n_per_cell, len(pool)), replace=False)
                rows.append(pool[idx][:, bins].mean(0))
            X.append(np.array(rows)); y.append(lab)
    X = np.array(X, float); y = np.array(y)
    good = ~np.isnan(X).any(axis=(0, 2))
    return X[:, good, :], y, bins

def temporal_generalization(df, train_contrast, test_contrast=None, backend=None,
                            n_pseudo=40, n_per_cell=8, seed=0, stride=2):
    \"\"\"Train at time t, test at t' -> (n_bins x n_bins) accuracy matrix (Fig 10).\"\"\"
    test_contrast = test_contrast or train_contrast
    dfa, dfb = _disjoint_halves(df, seed)
    Xtr, ytr, bins = build_pseudo_trials_tc(dfa, *train_contrast, n_pseudo=n_pseudo,
                                            n_per_cell=n_per_cell, seed=seed, stride=stride)
    Xte, yte, _ = build_pseudo_trials_tc(dfb, *test_contrast, n_pseudo=n_pseudo,
                                         n_per_cell=n_per_cell, seed=seed + 1, stride=stride)
    k = min(Xtr.shape[1], Xte.shape[1]); Xtr, Xte = Xtr[:, :k, :], Xte[:, :k, :]
    nb = len(bins); M = np.zeros((nb, nb))
    for i in range(nb):
        for j in range(nb):
            M[i, j] = (_fit_predict(Xtr[:, :, i], ytr, Xte[:, :, j], backend) == yte).mean()
    return dict(matrix=M, bins=bins)
""")

# ------------------------------- A5 hints/solutions -------------------------
_register("a5_hint1", "md", """
**A5 — hint 1 (the confound this defeats).** A bigger effect crosses any *absolute*
threshold sooner — that would make "earlier" just mean "larger". Normalizing to each
process's **own peak** kills a pure multiplicative amplitude difference: if
`stab(t) = k·flex(t)`, both cross 50%-of-peak at the *same* sample. That invariance is the
whole design — and a one-line unit test (scale a waveform by k, assert onset unchanged).
""")

_register("a5_hint2", "md", """
**A5 — hint 2 (mechanics).**
- `interaction_time_course`: attach cell labels with `_canonical_labels`; per electrode form
  the four `(cond,mod)` cell **mean time courses** and take the difference-of-differences per
  bin; average across electrodes. (Same d-o-d as `_interaction_cluster`, unthresholded.)
- `onset_50pct_peak`: `peak = max(effect*expected_sign)`; walk the rising flank to the first
  upward crossing of `0.5*peak`; linear-interpolate for sub-sample resolution.
- `jackknife_onset_difference`: measure onset on each **leave-one-subject-out** grand average;
  jackknife SE `= sqrt((N-1)/N * Σ(d_i - d̄)²)`; Ulrich–Miller `t_corrected = t_raw/(N-1)`.
""")

_register("a5_solution", "code", """
def interaction_time_course(df, key, contrast_mode="proportion", contrasts=None):
    \"\"\"Signed difference-of-differences per time bin for one process, elec-averaged.
    df['hg'] holds per-trial TIME COURSES; df.attrs['times'] the time vector.\"\"\"
    contrasts = finalize_contrasts(df, resolve_contrasts(contrast_mode, contrasts))
    work = _canonical_labels(df, contrasts)
    condcol, modcol = (("_scond", "_smod") if key == "stability" else ("_fcond", "_fmod"))
    times = df.attrs.get("times")
    per_elec = []
    for _, g in work.groupby(["subject", "electrode"]):
        hg = np.vstack([np.asarray(v, float) for v in g["hg"].to_numpy()])
        cond = g[condcol].to_numpy(); mod = g[modcol].to_numpy()
        valid = ~(np.isnan(cond) | np.isnan(mod))
        hg, cond, mod = hg[valid], cond[valid], mod[valid]
        cells, ok = {}, True
        for cv in (1., 0.):
            for mv in (1., 0.):
                sel = (cond == cv) & (mod == mv)
                if sel.sum() < 2: ok = False; break
                cells[(cv, mv)] = hg[sel].mean(0)
            if not ok: break
        if not ok: continue
        per_elec.append((cells[(1., 1.)] - cells[(0., 1.)])
                        - (cells[(1., 0.)] - cells[(0., 0.)]))
    effect = np.nanmean(np.vstack(per_elec), axis=0)
    if times is None: times = np.arange(len(effect))
    return np.asarray(times), effect

def onset_50pct_peak(times, effect, expected_sign=+1):
    \"\"\"First rising-flank crossing of 50% of the peak (amplitude-scale invariant).\"\"\"
    e = np.asarray(effect, float) * expected_sign
    peak_idx = int(np.nanargmax(e)); peak = e[peak_idx]
    if not np.isfinite(peak) or peak <= 0: return np.nan
    thr = 0.5 * peak
    if e[0] >= thr: return float(times[0])
    for i in range(1, peak_idx + 1):
        if e[i - 1] < thr <= e[i]:
            frac = (thr - e[i - 1]) / (e[i] - e[i - 1])
            return float(times[i - 1] + frac * (times[i] - times[i - 1]))
    return float(times[peak_idx])

def peak_latency(times, effect, expected_sign=+1):
    e = np.asarray(effect, float) * expected_sign
    return float(times[int(np.nanargmax(e))])

def jackknife_onset_difference(df_by_subject, expected_signs=(+1, +1),
                               contrast_mode="proportion"):
    \"\"\"Ulrich-Miller jackknifed LWPC-vs-LWPS onset difference over leave-one-out averages.\"\"\"
    subjects = sorted(df_by_subject["subject"].unique()); N = len(subjects)
    o_s, o_f = [], []
    for s in subjects:
        loo = df_by_subject[df_by_subject["subject"] != s]
        loo.attrs["times"] = df_by_subject.attrs.get("times")
        t, es = interaction_time_course(loo, "stability", contrast_mode)
        _, ef = interaction_time_course(loo, "flexibility", contrast_mode)
        o_s.append(onset_50pct_peak(t, es, expected_signs[0]))
        o_f.append(onset_50pct_peak(t, ef, expected_signs[1]))
    o_s, o_f = np.array(o_s), np.array(o_f)
    d = o_s - o_f; d_bar = np.nanmean(d)
    se = np.sqrt((N - 1) / N * np.nansum((d - d_bar) ** 2))     # jackknife SE
    t_raw = d_bar / (np.nanstd(d, ddof=1) / np.sqrt(N))
    t_corr = t_raw / (N - 1)                                    # Ulrich-Miller correction
    p = 2 * _tdist.sf(abs(t_corr), N - 1)
    t, es = interaction_time_course(df_by_subject, "stability", contrast_mode)
    _, ef = interaction_time_course(df_by_subject, "flexibility", contrast_mode)
    return dict(onset_lwpc=onset_50pct_peak(t, es, expected_signs[0]),
                onset_lwps=onset_50pct_peak(t, ef, expected_signs[1]),
                diff=float(d_bar), se=float(se), t_corrected=float(t_corr),
                p=float(p), ci=(float(d_bar - 1.96 * se), float(d_bar + 1.96 * se)))
""")

# ------------------------------- A6 hints/solutions -------------------------
_register("a6_hint1", "md", """
**A6 — hint 1 (two levels, very different power).** *Across subjects* (n = subjects) is
honest but underpowered — report it with its n and the caveat. *Within subject, single-trial*
is the preferred, high-power test: does trial-by-trial HG in the LWPC group predict the
trial-by-trial congruency-sequence RT adjustment? Always check the **cross pairing** (LWPC HG
↔ *switch* adjustment) is weaker — that's the specificity control.
""")

_register("a6_hint2", "md", """
**A6 — hint 2 (what to reach for).**
- `subject_level_brain_behavior`: reduce A1 labels to a per-subject neural summary
  (`groupby('subject')['S'].sum()`), merge with the behavioral magnitudes, `scipy.stats.pearsonr`.
- `trialwise_brain_behavior`: `statsmodels.formula.api.mixedlm('rt_adjustment ~ hg_group',
  trial_df, groups=trial_df['subject']).fit()`; read `.params` / `.pvalues`. The real
  behavioral magnitudes come from `stats/erin_linear_mixed_effects_model.py` / `combinedData.csv`,
  **not** recomputed from scratch.
""")

_register("a6_solution", "code", """
def subject_level_brain_behavior(elec_labels, behavior):
    \"\"\"Across-subject corr of neural selectivity counts vs behavioral LWPC/LWPS magnitude.
    behavior: DataFrame(subject, beh_lwpc, beh_lwps). UNDERPOWERED at n=subjects.\"\"\"
    g = elec_labels.groupby("subject")
    neural = pd.DataFrame({"subject": list(g.groups),
                           "n_S": g["S"].sum().values, "n_F": g["F"].sum().values})
    m = neural.merge(behavior, on="subject")
    cl, pl = pearsonr(m["n_S"], m["beh_lwpc"])
    cf, pf = pearsonr(m["n_F"], m["beh_lwps"])
    return dict(corr_lwpc=float(cl), p_lwpc=float(pl),
                corr_lwps=float(cf), p_lwps=float(pf), n_subjects=len(m),
                caveat="underpowered at n=subjects")

def trialwise_brain_behavior(trial_df, group="LWPC"):
    \"\"\"Within-subject single-trial mixed model: HG in a group -> matching RT adjustment.\"\"\"
    beh = "cong_seq_adj" if group == "LWPC" else "switch_adj"
    hgc = "hg_lwpc" if group == "LWPC" else "hg_lwps"
    m = smf.mixedlm(f"{beh} ~ {hgc}", trial_df, groups=trial_df["subject"]).fit()
    return dict(slope=float(m.params[hgc]), p=float(m.pvalues[hgc]), n_trials=len(trial_df))
""")

print("reveal registry loaded:")
list_reveals()


### Configuration — point this at your data
Edit the config below to match **your** environment. The loader tries the real epoched
HG data (clipped to LPFC) and **falls back to synthetic** if anything is missing, so the
rest of the notebook always runs.

- `LAB_root=None` auto-resolves the CoganLab root (Box on macOS/Windows, `/cwork/$USER`
  on the DCC). Set it explicitly to override.
- `EPOCHS_ROOT_FILE` is required for a real load — the same value the DCC submit script
  uses (e.g. `Stimulus_1sec_preStimulusBase_decFactor_10`). Left `None` → synthetic.
- `SUBJECTS` = your **one or two** subjects. LPFC electrodes are selected via the
  `lpfc` ROI in `src/analysis/config/rois.py`, using the same machinery as the pipeline.

In [ ]:
from types import SimpleNamespace

CONFIG = SimpleNamespace(
    LAB_root        = None,                       # None -> auto-resolve
    task            = "GlobalLocal",
    epochs_root_file= None,                       # e.g. "Stimulus_1sec_preStimulusBase_decFactor_10"
    subjects        = ["D0057", "D0059"],         # your 1-2 subjects
    window_tmin     = 0.0,                         # analysis window (s post-stimulus)
    window_tmax     = 0.5,
    acc_trials_only = True,
    # synthetic fallback knobs (used only if the real load fails):
    synth_n_subjects= 12,                          # enough subjects for A2 to be non-degenerate
    synth_seed      = 0,
)
CONFIG

#### The loader: real LPFC data → synthetic fallback
`load_sandbox_df(CONFIG)` returns the long-format single-trial table the analysis expects
(one row per electrode×trial): `subject, electrode, hg, congruency, switchType,
incongruent_proportion, switch_proportion`. It uses the **exact** production path
(`load_HG_ev1_rescaled_per_subject` → `resolve_electrodes_to_keep` on the `lpfc` ROI →
`assemble_long_df`). If that path raises (no Box/cluster, missing recon, etc.), it prints
why and returns a 12-subject synthetic table carrying the same columns and a real LWPC/LWPS
interaction, so A1/A2 have recoverable signal.

In [ ]:
def load_real_lpfc_df(cfg):
    """Production path: load 1-2 subjects, clip to LPFC, assemble the long df."""
    from src.analysis.utils.general_utils import (
        resolve_lab_root, resolve_electrodes_to_keep,
        load_HG_ev1_rescaled_per_subject,
    )
    from src.analysis.config.rois import rois_dict
    from dcc_scripts.stats.stability_flexibility_segregation_dcc import assemble_long_df

    if cfg.epochs_root_file is None:
        raise RuntimeError("CONFIG.epochs_root_file is None — set it for a real load.")

    LAB_root = resolve_lab_root(cfg.LAB_root)
    subjects_epochs = load_HG_ev1_rescaled_per_subject(
        subjects=cfg.subjects, epochs_root_file=cfg.epochs_root_file,
        task=cfg.task, LAB_root=LAB_root, acc_trials_only=cfg.acc_trials_only)

    # LPFC clip: reuse resolve_electrodes_to_keep with just the lpfc ROI.
    args = SimpleNamespace(subjects=cfg.subjects, task=cfg.task,
                           epochs_root_file=cfg.epochs_root_file,
                           rois_dict={"lpfc": rois_dict["lpfc"]},
                           electrodes="all")
    keep = resolve_electrodes_to_keep(args, LAB_root)     # {subject: {channels}}
    df = assemble_long_df(subjects_epochs, cfg.window_tmin, cfg.window_tmax,
                          electrodes_to_keep=keep, effect_measure="cohens_d")
    df.attrs["source"] = "real-lpfc"
    return df


def load_synth_df(cfg):
    """Fallback: ground-truth synthetic df with LWPC/LWPS interactions."""
    df = _synthetic_df(effect_measure="cohens_d", seed=cfg.synth_seed)
    # _synthetic_df emits 12 subjects; subset if the user asked for fewer *and* wants that.
    df.attrs["source"] = "synthetic"
    return df


def load_sandbox_df(cfg):
    try:
        df = load_real_lpfc_df(cfg)
        print(f"✓ loaded REAL LPFC data: {len(df)} rows | "
              f"{df.subject.nunique()} subjects | {df.electrode.nunique()} electrodes")
        return df
    except Exception as e:
        print(f"⚠ real load unavailable ({type(e).__name__}: {e})")
        print("  → falling back to SYNTHETIC ground-truth data.")
        df = load_synth_df(cfg)
        print(f"✓ synthetic: {len(df)} rows | {df.subject.nunique()} subjects | "
              f"{df.electrode.nunique()} electrodes")
        return df


df = load_sandbox_df(CONFIG)
print("data source:", df.attrs.get("source"))
df.head()

#### Sanity-look at the data you'll be analyzing
The proportion columns (`incongruent_proportion`, `switch_proportion`) are what make A1 an
*interaction* problem — note their deliberately unequal cell counts (~75/25), the exact
imbalance that makes Type III + sum coding necessary.

In [ ]:
print("columns:", list(df.columns))
print("\nsubjects:", sorted(df.subject.unique()))
print("electrodes per subject:")
print(df.groupby("subject")["electrode"].nunique())
print("\ncongruency × incongruent_proportion cell counts (per-trial rows, one electrode):")
one = df[df.electrode == df.electrode.iloc[0]]
display(pd.crosstab(one.congruency, one.incongruent_proportion))
print("switchType × switch_proportion:")
display(pd.crosstab(one.switchType, one.switch_proportion))

---
# A1 · Per-electrode two-way ANOVA electrode definition

**Goal.** Produce the *parametric, headline* electrode definition: for each electrode, the
**interaction** F/p of a two-way ANOVA — LWPC = `congruency × incongruent_proportion`,
LWPS = `switchType × switch_proportion` — then FDR across electrodes to set the `S`
(stability) and `F` (flexibility) flags. Output columns match `per_electrode_labels` so it's
a drop-in `labels` for `cmh_conjunction`.

**The one subtle thing:** Type III SS with **sum/effect coding**, or a pure main effect
leaks into the "interaction" under the unequal proportion cells. That's the whole
assignment.

### Task A1.1 — `_anova_interaction_stats` (one electrode)
Implement the stub. Then run the check cell.
*Hints:* `reveal("a1_stats_hint1")`, `reveal("a1_stats_hint2")`. *Answer:* `reveal("a1_stats_solution")` or `load_reference("a1_stats")`.

In [ ]:
def _anova_interaction_stats(elec_df, cond_col, mod_col, hg_col="hg"):
    """Fit ONE electrode's two-way Type III ANOVA; return {'F':..., 'p':...} for
    the interaction term (NaN,NaN on a singular fit).

    Steps: sum-coded formula  ->  smf.ols(...).fit()  ->  anova_lm(model, typ=3)
           ->  read the interaction row's 'F' and 'PR(>F)'  (try/except -> NaN)."""
    try:
        # Sum-code BOTH factors so Type III is a well-posed, equal-cell-weighted
        # model over the deliberately unequal proportion cells.
        formula = f"{hg_col} ~ C({cond_col}, Sum) * C({mod_col}, Sum)"
        model = smf.ols(formula, data=elec_df).fit()
        aov = anova_lm(model, typ=3)                       # Type III (margin-respecting)
        inter = f"C({cond_col}, Sum):C({mod_col}, Sum)"    # the interaction row
        return {"F": float(aov.loc[inter, "F"]),
                "p": float(aov.loc[inter, "PR(>F)"])}
    except Exception:
        return {"F": np.nan, "p": np.nan}                  # singular / missing cell

In [ ]:
# --- quick check on a single electrode -------------------------------------
_g = df[df.electrode == df.electrode.iloc[0]]
try:
    _st = _anova_interaction_stats(_g, "congruency", "incongruent_proportion")
    _fl = _anova_interaction_stats(_g, "switchType", "switch_proportion")
    print("stability (LWPC) interaction:", _st)
    print("flexibility (LWPS) interaction:", _fl)
    assert set(_st) == {"F", "p"}, "return a dict with keys F and p"
    print("✓ shape looks right")
except NotImplementedError as e:
    print("stub not implemented yet:", e)
    print("→ try it, or run:  reveal('a1_stats_hint1')  /  load_reference('a1_stats')")

### Task A1.2 — `per_electrode_anova_labels` (all electrodes → S/F table)
Loop electrodes, get interaction F/p for both processes, extract the **sign** from the
module's own `_interaction_effect` (so it matches §2), FDR across electrodes, set `S`/`F`,
and add the two **cross-interaction controls**.
*Hints:* `reveal("a1_labels_hint1")`, `reveal("a1_labels_hint2")`. *Answer:* `reveal("a1_labels_solution")` / `load_reference("a1_labels")`.

In [ ]:
def per_electrode_anova_labels(df, alpha=0.05, contrast_mode="proportion",
                               contrasts=None, include_cross_controls=True,
                               require_sign=False):
    """Parametric per-electrode S/F labels from the two-way interaction ANOVA.

    Return one row per electrode with at least:
        subject, electrode, p_cong, q_cong, S, p_switch, q_switch, F,
        s_sign, f_sign  (+ p_cross_cs, p_cross_si if include_cross_controls).

    Drop-in `labels` for cmh_conjunction (columns match per_electrode_labels)."""
    # 1. canonicalize once so the _scond/_smod/_fcond/_fmod cell labels exist
    contrasts = finalize_contrasts(df, resolve_contrasts(contrast_mode, contrasts))
    work = _canonical_labels(df, contrasts)               # attaches _scond/_smod/...
    recs = []
    for (subj, elec), g in work.groupby(["subject", "electrode"]):
        # 2. the two construct interactions (F is UNSIGNED)
        st = _anova_interaction_stats(g, "congruency", "incongruent_proportion")
        fl = _anova_interaction_stats(g, "switchType", "switch_proportion")
        # ...so borrow the SIGN from the module's own equal-cell-weight estimator,
        #    exactly the quantity the S2 continuous correlation uses.
        s_sign = np.sign(_interaction_effect(g["hg"].to_numpy(),
                                             g["_scond"].to_numpy(),
                                             g["_smod"].to_numpy(), "cohens_d", alpha))
        f_sign = np.sign(_interaction_effect(g["hg"].to_numpy(),
                                             g["_fcond"].to_numpy(),
                                             g["_fmod"].to_numpy(), "cohens_d", alpha))
        rec = dict(subject=subj, electrode=elec,
                   p_cong=st["p"], F_cong=st["F"], s_sign=s_sign,
                   p_switch=fl["p"], F_switch=fl["F"], f_sign=f_sign)
        if include_cross_controls:                        # specificity controls (report only)
            rec["p_cross_cs"] = _anova_interaction_stats(
                g, "congruency", "switch_proportion")["p"]
            rec["p_cross_si"] = _anova_interaction_stats(
                g, "switchType", "incongruent_proportion")["p"]
        recs.append(rec)
    out = pd.DataFrame(recs)
    # 3. FDR across electrodes -> q; flags at alpha (optionally sign-gated)
    out["q_cong"]   = multipletests(out["p_cong"].fillna(1),   method="fdr_bh")[1]
    out["q_switch"] = multipletests(out["p_switch"].fillna(1), method="fdr_bh")[1]
    S = out["q_cong"] < alpha
    F = out["q_switch"] < alpha
    if require_sign:                                      # keep only predicted-direction growth
        S = S & (out["s_sign"] > 0)
        F = F & (out["f_sign"] > 0)
    out["S"] = S.astype(int)
    out["F"] = F.astype(int)
    return out

#### Acceptance check A1
The assignment's success criteria, made concrete:
1. **Column compatibility** — `cmh_conjunction(anova_labels)` runs unchanged.
2. **Agreement** with the nonparametric `per_electrode_labels` (same balanced interaction,
   different estimator) — high on synthetic ground truth (Cohen's κ well above 0).
3. **Cross-controls near-null** — no true cross-effect in the generator, so `p_cross_*` is
   ~uniform (rejection rate near α).

In [ ]:
# Build the ANOVA labels, then validate.
anova_labels = per_electrode_anova_labels(df, contrast_mode="proportion")
display(anova_labels.head())

# 1. drop-in compatibility with the existing CMH
conj = cmh_conjunction(anova_labels)
print(f"[1] cmh_conjunction accepts ANOVA labels — MH OR = {conj['mh_odds_ratio']:.3f}")

# 2. agreement vs the nonparametric labels (small n_perm for speed)
nonpar = per_electrode_labels(df, n_perm=200, contrast_mode="proportion")
m = anova_labels.merge(nonpar[["electrode", "S", "F"]], on="electrode",
                       suffixes=("_anova", "_nonpar"))
def _kappa(a, b):
    po = (a == b).mean()
    pe = a.mean()*b.mean() + (1-a.mean())*(1-b.mean())
    return (po - pe) / (1 - pe) if (1 - pe) > 0 else np.nan
for col in ["S", "F"]:
    a, b = m[f"{col}_anova"], m[f"{col}_nonpar"]
    print(f"[2] {col}: agreement={(a==b).mean():.3f}  kappa={_kappa(a,b):+.3f}  "
          f"(anova n={int(a.sum())}, nonpar n={int(b.sum())})")

# 3. cross-control specificity (report only)
if "p_cross_cs" in anova_labels:
    for c in ["p_cross_cs", "p_cross_si"]:
        p = anova_labels[c].dropna()
        print(f"[3] {c}: mean={p.mean():.3f}  frac<0.05={np.mean(p<0.05):.3f}  "
              f"(want ~uniform / ~0.05)")

In [ ]:
# --- visualize the ANOVA labels --------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
sc = ax[0].scatter(anova_labels["q_cong"], anova_labels["q_switch"],
                   c=anova_labels["S"] + 2*anova_labels["F"], cmap="viridis",
                   s=22, alpha=.75)
ax[0].axvline(0.05, ls="--", c="k", lw=1); ax[0].axhline(0.05, ls="--", c="k", lw=1)
ax[0].set(xlabel="q (stability / LWPC)", ylabel="q (flexibility / LWPS)",
          title="FDR q-values (dashed = α)", xlim=(-.02, 1.02), ylim=(-.02, 1.02))

both = int(((anova_labels.S==1)&(anova_labels.F==1)).sum())
so   = int(((anova_labels.S==1)&(anova_labels.F==0)).sum())
fo   = int(((anova_labels.S==0)&(anova_labels.F==1)).sum())
ne   = int(((anova_labels.S==0)&(anova_labels.F==0)).sum())
ax[1].bar(["neither","S only","F only","both"], [ne, so, fo, both],
          color=["#ccc", "#2c7fb8", "#d95f0e", "#31a354"])
ax[1].set(title="Electrode selectivity classes (ANOVA)", ylabel="# electrodes")
fig.tight_layout(); plt.show()

---
# A2 · Conjunction permutation null + threshold sweep

`cmh_conjunction` already gives the MH odds ratio and its parametric tests. A2 adds two
robustness pieces the plan asks for:
- a **within-subject permutation null** on the "both" count (respects subject nesting
  better than the analytic hypergeometric), and
- a **threshold sweep** so a segregation claim is shown stable across α, not an artifact of
  one cutoff.

Both are thin wrappers — **reuse** `cmh_conjunction`, don't reimplement the CMH.

### Task A2.1 — `conjunction_permutation_null`
Shuffle `F` **within each subject** (S fixed), recount the overlap, build the null.
*Hints:* `reveal("a2_null_hint1")`, `reveal("a2_null_hint2")`. *Answer:* `reveal("a2_null_solution")` / `load_reference("a2_null")`.

In [ ]:
def conjunction_permutation_null(labels, n_perm=10000, seed=0):
    """Empirical null for the count of 'both' (S==1 & F==1) electrodes.

    Return dict(observed:int, null:ndarray(n_perm), p_two_sided:float, z:float).
    Shuffle F WITHIN each subject only (groupby subject), so each subject's S.sum()
    and F.sum() stay fixed and only the pairing is randomized (the CMH null)."""
    lab = labels.dropna(subset=["S", "F"]).copy()
    S = lab["S"].to_numpy().astype(int)
    F = lab["F"].to_numpy().astype(int)
    subj = lab["subject"].to_numpy()
    groups = [np.where(subj == s)[0] for s in np.unique(subj)]   # per-subject row indices
    observed = int(((S == 1) & (F == 1)).sum())
    rng = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for i in range(n_perm):
        Fp = F.copy()
        for idx in groups:                 # shuffle F WITHIN each subject only
            Fp[idx] = F[rng.permutation(idx)]
        null[i] = int(((S == 1) & (Fp == 1)).sum())
    mean = null.mean()
    p = (np.sum(np.abs(null - mean) >= abs(observed - mean)) + 1) / (n_perm + 1)
    z = (observed - mean) / null.std() if null.std() > 0 else np.nan
    return dict(observed=observed, null=null, p_two_sided=float(p), z=float(z))

#### Acceptance check A2.1
- **Invariant:** every permutation preserves each subject's `S.sum()` and `F.sum()`
  (only the pairing moves). We assert it directly on the null machinery.
- **Behavior on independent data:** the synthetic generator draws each electrode's
  stability/flexibility sensitivities independently, so the observed overlap should sit
  *inside* the null (p not significant, z near 0).

In [ ]:
null_res = conjunction_permutation_null(anova_labels, n_perm=3000, seed=1)
print(f"observed 'both' = {null_res['observed']} | "
      f"null mean = {null_res['null'].mean():.2f} | "
      f"p(two-sided) = {null_res['p_two_sided']:.3f} | z = {null_res['z']:+.2f}")

# --- assert the within-subject invariant directly --------------------------
_lab = anova_labels.dropna(subset=['S','F']).copy()
_S = _lab['S'].to_numpy(); _F = _lab['F'].to_numpy(); _subj = _lab['subject'].to_numpy()
_groups = [np.where(_subj==s)[0] for s in np.unique(_subj)]
_rng = np.random.default_rng(0)
_S_by_subj = _lab.groupby('subject')['S'].sum()
_F_by_subj = _lab.groupby('subject')['F'].sum()
for _ in range(50):
    _Fp = _F.copy()
    for idx in _groups:
        _Fp[idx] = _F[_rng.permutation(idx)]
    perm = _lab.assign(Fp=_Fp)
    assert (perm.groupby('subject')['S'].sum() == _S_by_subj).all()
    assert (perm.groupby('subject')['Fp'].sum() == _F_by_subj).all()
print("✓ invariant holds: per-subject S and F counts unchanged under permutation")

In [ ]:
# --- visualize the null ----------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_res["null"], bins=30, color="#bbb", edgecolor="w")
ax.axvline(null_res["observed"], color="#d7191c", lw=2.5,
           label=f"observed = {null_res['observed']}")
ax.set(title=f"Within-subject permutation null for 'both' count "
             f"(p={null_res['p_two_sided']:.3f})",
       xlabel="# 'both' electrodes under null", ylabel="# permutations")
ax.legend(); fig.tight_layout(); plt.show()

### Task A2.2 — `conjunction_threshold_sweep`
Recompute the counts + MH OR across a range of selection thresholds (here, q-value cutoffs
on the ANOVA labels). Loop, call `cmh_conjunction`, return a tidy table.
*Hint:* `reveal("a2_sweep_hint1")`. *Answer:* `reveal("a2_sweep_solution")` / `load_reference("a2_sweep")`.

In [ ]:
def conjunction_threshold_sweep(labels_by_threshold, thresholds):
    """Recompute overlap OR + counts across selection thresholds.

    labels_by_threshold : callable  threshold -> labels DataFrame (S/F recomputed).
    Return a tidy DataFrame: threshold, n_S, n_F, n_both, mh_odds_ratio, cmh_p."""
    rows = []
    for t in thresholds:
        lab = labels_by_threshold(t)                 # fresh S/F at this threshold
        res = cmh_conjunction(lab)                   # reuse the existing CMH machinery
        rows.append(dict(threshold=t,
                         n_S=int(lab.S.sum()), n_F=int(lab.F.sum()),
                         n_both=int(((lab.S == 1) & (lab.F == 1)).sum()),
                         mh_odds_ratio=res["mh_odds_ratio"],
                         cmh_p=res["cmh"].pvalue))
    return pd.DataFrame(rows)

#### Acceptance check A2.2
On the independent synthetic data the OR should **hover near 1 across the whole sweep** and
the conclusion (segregated vs. shared) should not flip at any reasonable threshold. On real
LPFC data, a sweep that stays on one side of 1 is the robustness evidence the plan wants;
one that flips is a *finding to report, not hide*.

In [ ]:
# Threshold on the ANOVA q-values: S/F = (q < cutoff).
def labels_at_qcut(qcut):
    l = anova_labels.copy()
    l["S"] = (l["q_cong"]   < qcut).astype(int)
    l["F"] = (l["q_switch"] < qcut).astype(int)
    return l

thresholds = [0.01, 0.05, 0.10, 0.20, 0.35, 0.50]
sweep = conjunction_threshold_sweep(labels_at_qcut, thresholds)
display(sweep)

# cross-check: the continuous correlation should tell the same story at the sweep midpoint
elec = add_responsiveness(
    compute_sensitivities(df, n_splits=30, contrast_mode="proportion"), df)
cont = prepare_continuous(elec)
corr = subject_clustered_corr(cont, n_perm=1000)
print(f"\ncontinuous corr = {corr['corr']:+.3f} (p={corr['p']:.3f})  "
      f"[>0 shared core, ≤0 segregated] — compare to OR vs 1 in the sweep")

In [ ]:
# --- plot OR vs threshold ---------------------------------------------------
fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(sweep["threshold"], sweep["mh_odds_ratio"], "o-", color="#31a354", lw=2,
         label="MH odds ratio")
ax1.axhline(1.0, ls="--", c="k", lw=1, label="OR = 1 (independence)")
ax1.set(xlabel="selection threshold (q-value cutoff)", ylabel="MH odds ratio")
ax1.set_title("Conjunction OR across thresholds\n(OR<1 → segregation, OR>1 → shared core)")
ax2 = ax1.twinx()
ax2.bar(sweep["threshold"], sweep["n_both"], width=0.015, alpha=.25, color="#2c7fb8")
ax2.set_ylabel("# 'both' electrodes", color="#2c7fb8")
ax1.legend(loc="upper left"); fig.tight_layout(); plt.show()

---
# A3 · Anatomy — ROI join + coverage-conditioned enrichment

**Goal.** "Are the distinct subpopulations in different *places*?" Join the selectivity
labels to each electrode's ROI, plot ROI-membership per group, and test whether group
membership is associated with ROI — **conditioned on coverage**, because iEEG coverage is
clinical and a raw ROI difference can just reflect *where electrodes are*.

*Drop-in target:* stats in a new `stability_flexibility_anatomy.py`; the brain surface plot
reuses the existing `src/analysis/vis/` Glasser renderer.

> The section feeds on **per-electrode S/F labels**. To stay independent of your A1
> implementation it uses the module's `per_electrode_labels`, but `per_electrode_anova_labels`
> from A1 is interchangeable here.

*Hints:* `reveal("a3_hint1")`, `reveal("a3_hint2")`. *Answer:* `reveal("a3_solution")` / `load_reference("a3")`.

In [ ]:
# --- labels + an electrode->ROI map (real localization, else a sandbox stand-in) ---
base_labels = per_electrode_labels(df, n_perm=150, contrast_mode="proportion")

def load_roi_map(labels, cfg):
    """Real electrode->ROI via the localization pipeline; else a synthetic LPFC map."""
    try:
        from src.analysis.utils.general_utils import (
            resolve_lab_root, make_or_load_subjects_electrodes_to_ROIs_dict)
        from src.analysis.config.rois import rois_dict
        LAB_root = resolve_lab_root(cfg.LAB_root)
        cfgdir = os.path.join(_root, "src", "analysis", "config")
        d = make_or_load_subjects_electrodes_to_ROIs_dict(
            subjects=cfg.subjects, task=cfg.task, LAB_root=LAB_root, save_dir=cfgdir)
        e2r = {}
        for sub, dd in d.items():
            for ch, roi in dd["default_dict"].items():
                e2r[f"{sub}-{ch}"] = roi
        mapped = labels["electrode"].map(e2r)
        if mapped.notna().mean() < 0.5:
            raise RuntimeError("real ROI map covers <50% of electrodes")
        print("✓ real electrode->ROI localization")
        return e2r
    except Exception as e:
        print(f"⚠ real ROI map unavailable ({type(e).__name__}: {e}) → synthetic LPFC map")
        LPFC = ["G_front_middle", "G_front_sup", "S_front_inf", "S_front_middle",
                "G_front_inf-Triangul", "G_front_inf-Opercular"]
        rng = np.random.default_rng(3)
        def _grp(s, f):
            return ("both" if s == 1 and f == 1 else "S_only" if s == 1
                    else "F_only" if f == 1 else "neither")
        rois = []
        for s, f in zip(labels["S"], labels["F"]):
            if _grp(s, f) == "both":                 # plant a coverage-independent enrichment
                rois.append(rng.choice(LPFC, p=[.72, .056, .056, .056, .056, .056]))
            else:
                rois.append(rng.choice(LPFC))
        return dict(zip(labels["electrode"], rois))

roi_map = load_roi_map(base_labels, CONFIG)

In [ ]:
# Task A3.1-3: attach_roi, build_coverage_matrix, roi_group_enrichment_test
def attach_roi(labels, electrodes_to_rois):
    """Add 'roi' (electrode->region) and a 4-way 'group' in {both,S_only,F_only,neither}."""
    raise NotImplementedError("A3.1 — join labels to ROI + derive the 4-way group")

def build_coverage_matrix(labels_with_roi):
    """subject x ROI boolean: does subject have ANY electrode in that ROI?"""
    raise NotImplementedError("A3.2 — subject x ROI coverage matrix")

def roi_group_enrichment_test(labels_with_roi, coverage, min_subjects=3,
                              n_perm=5000, seed=0, groups_keep=("S_only", "F_only", "both")):
    """Coverage-conditioned test: restrict to ROIs in >= min_subjects, chi-square on the
    group x ROI table, null by permuting group WITHIN subject. Return dict incl. p +
    per_roi_coverage."""
    raise NotImplementedError("A3.3 — coverage-conditioned ROI enrichment test")

#### Acceptance check A3
- Every anatomical claim is **conditioned on coverage** — the test drops ROIs below the
  *k*-subject threshold and reports how many subjects cover each retained ROI.
- The **within-subject** group permutation respects nesting: the null shuffles group labels
  only inside each subject's electrodes.

In [ ]:
lab_roi = attach_roi(base_labels, roi_map)
cov = build_coverage_matrix(lab_roi)
print("coverage matrix (subject x ROI):", cov.shape)

enr = roi_group_enrichment_test(lab_roi, cov, min_subjects=3, n_perm=2000)
print(f"\nROIs tested (coverage ≥ 3 subjects): {enr['rois_tested']}")
print("per-ROI coverage (n subjects):")
print(enr["per_roi_coverage"].to_string())
print(f"\nchi2 = {enr['observed_stat']:.2f}   permutation p = {enr['p']:.4f}")
print("contingency (group x ROI):")
display(enr["contingency"])

In [ ]:
# ROI-membership histogram per selectivity group (works without a brain surface)
grp_counts = (lab_roi[lab_roi["group"] != "neither"]
              .groupby(["group", "roi"]).size().unstack(fill_value=0))
grp_counts = grp_counts[enr["rois_tested"]]        # coverage-conditioned ROIs
fig, ax = plt.subplots(figsize=(10, 4.2))
grp_counts.T.plot(kind="bar", ax=ax,
                  color={"S_only": "#2c7fb8", "F_only": "#d95f0e", "both": "#31a354"})
ax.set(title="ROI membership by selectivity group (coverage-conditioned ROIs)",
       ylabel="# electrodes", xlabel="ROI")
ax.tick_params(axis="x", labelrotation=30); fig.tight_layout(); plt.show()

**Brain-surface plot (real-data only).** `plot_selectivity_groups_on_brain` is a thin
wrapper over the existing Glasser SVG renderer in `src/analysis/vis/`
(`brain_figure_glasser_separate_svgs_lateral_medial_view_less_bold.py`) — pass your per-group
electrode lists as its highlight sets. It needs MNE + PyVista + real recon coordinates, so it
isn't runnable in this sandbox; the histogram above is the coverage-safe substitute. The
reference wrapper is in the assignment skeleton (`docs/skeletons/a3_anatomy.py`).

---
# A4 · Cross-decoding — pseudo-trials, label transfer, mean-removal control

**Goal.** Co-localization ≠ shared *code*. Train a decoder on one contrast and test it on
another: shared representational geometry → transfer works; orthogonal codes → it fails. The
two **non-negotiable** controls (plan §0.8): **disjoint train/test trials** (no circularity)
and **per-condition mean removal** (a cross-effect that vanishes after mean removal was a
univariate offset, not a code).

> **Backend.** `cross_decode` runs through a **pluggable classifier backend**: the project's
> real `Decoder` (PCA + LDA, from `src/analysis/decoding/decoding.py`) when it's importable, or
> a compact `LogisticRegression` fallback so the sandbox runs even without the `ieeg`/`mne`
> stack. Same pseudo-trial + disjoint-split + null discipline either way — the production
> drop-in (`src/analysis/decoding/cross_decoding.py`) simply pins `backend='decoder'`.
> This section covers the label-transfer core **plus** the within-block baseline (Fig 9) and
> temporal generalization (Fig 10); set-transfer (design b) remains a short exercise.

*Hints:* `reveal("a4_hint1")`, `reveal("a4_hint2")`. *Answer:* `reveal("a4_solution")` / `load_reference("a4")`.

In [ ]:
# --- pick the classifier backend: real project Decoder if available, else compact ---
try:
    from src.analysis.decoding.decoding import Decoder
    _HAS_DECODER = True
except Exception as e:
    _HAS_DECODER = False
    print(f"project Decoder unavailable ({type(e).__name__}: {e}) — compact classifier only")

# importing ieeg/mne hijacks the matplotlib backend (→ Agg), which kills inline
# figures for the plots below. Restore inline rendering.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

# 'auto' -> Decoder if importable else sklearn; force with 'decoder' / 'sklearn'
A4_BACKEND = "auto"
_resolved = ("decoder" if (A4_BACKEND == "decoder" or (A4_BACKEND == "auto" and _HAS_DECODER))
             else "sklearn")
print(f"Decoder importable: {_HAS_DECODER} | A4_BACKEND={A4_BACKEND!r} -> using {_resolved!r}")

# For speed on the synthetic fallback (hundreds of electrodes), sub-sample the
# pseudopopulation; real LPFC data (few electrodes) is used whole.
_elecs = sorted(df["electrode"].unique())
if len(_elecs) > 90:
    _keep = set(np.random.default_rng(0).choice(_elecs, 90, replace=False))
    df_a4 = df[df["electrode"].isin(_keep)].copy()
    print(f"A4 uses a {df_a4.electrode.nunique()}-electrode pseudopopulation (subsampled)")
else:
    df_a4 = df
    print(f"A4 uses all {df_a4.electrode.nunique()} electrodes")

In [ ]:
# Task A4.1-4
def _fit_predict(Xtr, ytr, Xte, backend=None, explained_variance=0.8, random_state=0):
    """Train on (Xtr,ytr), predict Xte, via a pluggable backend. 'decoder' -> project
    Decoder (PCA+LDA); 'sklearn' -> compact pipeline; None/'auto' -> the A4_BACKEND global."""
    raise NotImplementedError("A4.1 — backend fit/predict (wraps Decoder or LogisticRegression)")

def build_pseudo_trials(df, label_col, pos, neg, n_pseudo=60, n_per_cell=8, seed=0):
    """Pseudopopulation pseudo-trials (features=electrodes; one row = per-electrode mean of
    n_per_cell sampled trials of a class). Return (X, y)."""
    raise NotImplementedError("A4.2 — pseudo-trial construction")

def remove_condition_means(X, condition_labels):
    """Subtract each condition's per-feature mean (the univariate-offset control)."""
    raise NotImplementedError("A4.3 — per-condition mean removal")

def cross_decode(df, train_spec, test_spec, strip_condition_means=False, backend=None,
                 n_pseudo=80, n_per_cell=8, n_perm=200, seed=0):
    """Train on train_spec contrast, TEST on test_spec (disjoint trials). spec =
    (label_col, pos, neg). Return dict(accuracy, null_mean, p). See reveal for the disjoint-
    half helper `_disjoint_halves` (uses `_fit_predict` for the backend)."""
    raise NotImplementedError("A4.4 — cross-condition label transfer")

#### Acceptance check A4 (the §0.8 confound controls)
- **`remove_condition_means`** zeroes each class's per-feature mean (unit test).
- **Circularity:** train and test pseudo-trials come from disjoint trial halves.
- **The key result:** on the synthetic generator stability (`bx`) and flexibility (`by`) are
  drawn *independently*, so within-contrast decoding beats chance while the **cross-decode is
  a univariate offset** — it looks above chance but **collapses to chance after
  `remove_condition_means`**. That collapse is the correct "segregated codes" verdict.

In [ ]:
if not _HAS_SKLEARN:
    print("sklearn unavailable — skipping A4 execution.")
else:
    # unit test: mean removal
    _X = np.random.default_rng(0).normal(size=(20, 5)); _y = np.array([0]*10 + [1]*10)
    _Xr = remove_condition_means(_X, _y)
    assert np.allclose(_Xr[_y==0].mean(0), 0) and np.allclose(_Xr[_y==1].mean(0), 0)
    print("✓ remove_condition_means unit test passed (per-class means → 0)\n")

    STAB = ("congruency", "i", "c")     # stability contrast
    FLEX = ("switchType", "s", "r")     # flexibility contrast
    within_s = cross_decode(df_a4, STAB, STAB, n_perm=100)
    within_f = cross_decode(df_a4, FLEX, FLEX, n_perm=100)
    cross    = cross_decode(df_a4, STAB, FLEX, n_perm=100)
    cross_mr = cross_decode(df_a4, STAB, FLEX, strip_condition_means=True, n_perm=100)
    print(f"within-stability   acc={within_s['accuracy']:.2f}  (null {within_s['null_mean']:.2f}, p={within_s['p']:.3f})")
    print(f"within-flexibility acc={within_f['accuracy']:.2f}  (null {within_f['null_mean']:.2f}, p={within_f['p']:.3f})")
    print(f"CROSS (train stab→test flex)      acc={cross['accuracy']:.2f}  (null {cross['null_mean']:.2f}, p={cross['p']:.3f})")
    print(f"CROSS + per-condition mean removal acc={cross_mr['accuracy']:.2f}  (p={cross_mr['p']:.3f})")
    print("→ cross survives? ", "yes (shared code)" if cross_mr['p'] < 0.05 else "NO → univariate offset, codes are segregated")

In [ ]:
if _HAS_SKLEARN:
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    names = ["within\nstability", "within\nflexibility", "cross\n(raw)", "cross\n(mean-removed)"]
    accs  = [within_s['accuracy'], within_f['accuracy'], cross['accuracy'], cross_mr['accuracy']]
    cols  = ["#2c7fb8", "#d95f0e", "#8856a7", "#c7c7c7"]
    ax.bar(names, accs, color=cols)
    ax.axhline(0.5, ls="--", c="k", lw=1, label="chance")
    for i, v in enumerate(accs): ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
    ax.set(title="Decoding accuracy: cross-effect is a univariate offset\n"
                 "(collapses to chance after per-condition mean removal)",
           ylabel="accuracy", ylim=(0, 1.05)); ax.legend()
    fig.tight_layout(); plt.show()

### A4 · Within-block decoding baseline (Fig 9)
Decode a contrast **within each block level** and compare accuracies — the decoding analog of
the univariate LWPC/LWPS effect, and where the neural cross-effect surfaces: congruency should
decode **better in mostly-incongruent blocks**.
*Hint:* `reveal("a4_within_hint")`. *Answer:* `reveal("a4_within_solution")` / `load_reference("a4_within")`.

In [ ]:
# Task A4.5
def within_block_decoding_baseline(df, contrast_spec, block_col, backend=None,
                                   n_pseudo=60, n_per_cell=3, n_perm=100, seed=0):
    """Decode contrast_spec WITHIN each level of block_col; compare accuracies (Fig 9).
    Return dict(per_block={level: cross_decode result}, block_levels, acc_difference)."""
    raise NotImplementedError("A4.5 — within-block decoding baseline (Fig 9)")

In [ ]:
if _HAS_SKLEARN:
    # few electrodes so decoding isn't at ceiling and the block modulation is visible
    _wb_el = sorted(df["electrode"].unique())
    if len(_wb_el) > 12:
        _wk = set(np.random.default_rng(1).choice(_wb_el, 10, replace=False))
        df_wb = df[df["electrode"].isin(_wk)].copy()
    else:
        df_wb = df
    # compact backend here: PCA+LDA pools the few electrodes to ceiling in BOTH blocks,
    # which hides the modulation; the compact classifier keeps accuracy off-ceiling so the
    # LWPC ordering is visible. On real LPFC data (more electrodes) either backend works.
    wb = within_block_decoding_baseline(df_wb, ("congruency", "i", "c"),
                                        "incongruent_proportion", backend="sklearn",
                                        n_pseudo=50, n_per_cell=2, n_perm=60)
    for b, r in wb["per_block"].items():
        print(f"  incongruent_proportion={b:g}: congruency decode acc={r['accuracy']:.2f} (p={r['p']:.3f})")
    print(f"→ Δacc(high − low block) = {wb['acc_difference']:+.2f}  "
          f"(neural LWPC: congruency decodes better in mostly-incongruent blocks)")

In [ ]:
if _HAS_SKLEARN:
    fig, ax = plt.subplots(figsize=(5.5, 4))
    lv = wb["block_levels"]; accs = [wb["per_block"][b]["accuracy"] for b in lv]
    ax.bar([f"{b:g}%\nincong." for b in lv], accs, color=["#9ecae1", "#2c7fb8"])
    ax.axhline(0.5, ls="--", c="k", lw=1, label="chance")
    for i, v in enumerate(accs): ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
    ax.set(title="Within-block congruency decoding (Fig 9)\n"
                 "higher in mostly-incongruent blocks = neural LWPC",
           ylabel="accuracy", ylim=(0, 1.05)); ax.legend()
    fig.tight_layout(); plt.show()

### A4 · Temporal generalization (Fig 10)
Train at time *t*, test at *t′*, for all pairs → a train×test accuracy matrix. A broad
off-diagonal block = a **sustained/stable** code; a narrow diagonal = a **moving/phasic** code.
Needs time-course pseudo-trials. *(Demo uses a compact cluster-mode synthetic; for real LPFC
time courses pass the `effect_measure='cluster'` df from A5's `load_timing_df(CONFIG)`.)*
*Hint:* `reveal("a4_tempgen_hint")`. *Answer:* `reveal("a4_tempgen_solution")` / `load_reference("a4_tempgen")`.

In [ ]:
# Task A4.6
def build_pseudo_trials_tc(df, label_col, pos, neg, n_pseudo=40, n_per_cell=8, seed=0, stride=2):
    """Time-course pseudo-trials: X (n_pseudo, n_elec, n_binsub), y, kept bin indices."""
    raise NotImplementedError("A4.6a — time-course pseudo-trials")

def temporal_generalization(df, train_contrast, test_contrast=None, backend=None,
                            n_pseudo=40, n_per_cell=8, seed=0, stride=2):
    """Train at time t, test at t' -> (n_bins x n_bins) accuracy matrix (Fig 10)."""
    raise NotImplementedError("A4.6b — temporal generalization matrix")

In [ ]:
if _HAS_SKLEARN:
    from src.analysis.stats.stability_flexibility_segregation import _synthetic_df as _sd
    _tc = _sd(effect_measure="cluster", n_time=16, seed=0)
    _kc = set(np.random.default_rng(1).choice(sorted(_tc.electrode.unique()), 60, replace=False))
    _tc = _tc[_tc.electrode.isin(_kc)].copy()
    tg = temporal_generalization(_tc, ("congruency", "i", "c"), n_pseudo=30, stride=2)
    M = tg["matrix"]
    print(f"train×test matrix {M.shape}: diagonal mean={np.mean(np.diag(M)):.2f}, "
          f"off-diagonal mean={np.mean(M[~np.eye(len(M), dtype=bool)]):.2f}")
    print("(effect is injected in the middle-half window → a sustained block there)")

In [ ]:
if _HAS_SKLEARN:
    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    im = ax.imshow(M, origin="lower", cmap="magma", vmin=0.4, vmax=1.0, aspect="equal")
    ax.set(title="Temporal generalization (Fig 10)\ncongruency code: train t × test t'",
           xlabel="test time bin", ylabel="train time bin")
    plt.colorbar(im, ax=ax, label="accuracy"); fig.tight_layout(); plt.show()

---
# A5 · Timing — interaction time course, 50%-of-peak onset, jackknife

**Goal.** A *sequence* claim neither overlap nor decoding can make: does stability
information arise **earlier** than flexibility (or vice versa)? The design defeats the
**latency–amplitude confound** — a bigger effect crosses any absolute threshold sooner, so
onset is defined as the first rising crossing of **50% of each process's own peak**. If
`stab(t) = k·flex(t)`, both cross at the same sample.

*Drop-in target:* this **is** the new `src/analysis/stats/stability_flexibility_timing.py`.

> A5 needs per-trial **time courses** (`effect_measure='cluster'` shape), not window means, so
> it loads its own dataset — real cluster-mode LPFC data if available, else a synthetic
> generator with LWPC planted to rise *earlier* than LWPS so the demo shows a real difference.

*Hints:* `reveal("a5_hint1")`, `reveal("a5_hint2")`. *Answer:* `reveal("a5_solution")` / `load_reference("a5")`.

In [ ]:
def _timing_synthetic_df(n_subj=10, n_elec=12, n_time=40, T_sec=0.6,
                         onset_lwpc=8, onset_lwps=18, seed=0):
    """Synthetic time-course table with LWPC ramping earlier than LWPS."""
    rng = np.random.default_rng(seed)
    times = np.linspace(0, T_sec, n_time)
    frames = []
    for s in range(n_subj):
        n_tr = int(rng.integers(200, 320))
        cong = rng.choice(["c", "i"], n_tr); sw = rng.choice(["s", "r"], n_tr)
        inc = rng.choice([25., 75.], n_tr); swp = rng.choice([25., 75.], n_tr)
        for e in range(n_elec):
            gain = rng.lognormal(0, 0.3)
            bx = abs(rng.normal(0.5, 0.2)); by = abs(rng.normal(0.5, 0.2))
            wl = np.zeros(n_time); wl[onset_lwpc:] = 1.0
            wf = np.zeros(n_time); wf[onset_lwps:] = 1.0
            lwpc_amp = bx * (cong == "i") * (inc == 75.)
            lwps_amp = by * (sw == "s") * (swp == 75.)
            tc = rng.normal(0, 1, (n_tr, n_time)) * gain
            tc += gain * (lwpc_amp[:, None]*wl[None, :] + lwps_amp[:, None]*wf[None, :])
            col = np.empty(n_tr, dtype=object)
            for i in range(n_tr): col[i] = tc[i]
            fr = pd.DataFrame(dict(subject=f"S{s:02d}", electrode=f"S{s:02d}-e{e}",
                                   congruency=cong, switchType=sw,
                                   incongruent_proportion=inc, switch_proportion=swp))
            fr["hg"] = col
            frames.append(fr)
    d = pd.concat(frames, ignore_index=True); d.attrs["times"] = times
    return d

def load_timing_df(cfg):
    """Real cluster-mode LPFC time courses if available, else the synthetic generator."""
    try:
        from src.analysis.utils.general_utils import (
            resolve_lab_root, resolve_electrodes_to_keep, load_HG_ev1_rescaled_per_subject)
        from src.analysis.config.rois import rois_dict
        from dcc_scripts.stats.stability_flexibility_segregation_dcc import assemble_long_df
        if cfg.epochs_root_file is None:
            raise RuntimeError("epochs_root_file is None")
        LAB_root = resolve_lab_root(cfg.LAB_root)
        se = load_HG_ev1_rescaled_per_subject(cfg.subjects, cfg.epochs_root_file, cfg.task,
                                              LAB_root=LAB_root, acc_trials_only=cfg.acc_trials_only)
        args = SimpleNamespace(subjects=cfg.subjects, task=cfg.task,
                               epochs_root_file=cfg.epochs_root_file,
                               rois_dict={"lpfc": rois_dict["lpfc"]}, electrodes="all")
        keep = resolve_electrodes_to_keep(args, LAB_root)
        d = assemble_long_df(se, cfg.window_tmin, cfg.window_tmax,
                             electrodes_to_keep=keep, effect_measure="cluster")
        # attach a times vector spanning the window
        T = len(np.asarray(d["hg"].iloc[0]))
        d.attrs["times"] = np.linspace(cfg.window_tmin, cfg.window_tmax, T)
        print(f"✓ real cluster-mode LPFC time courses: {d.electrode.nunique()} electrodes, T={T}")
        return d
    except Exception as e:
        print(f"⚠ real timing data unavailable ({type(e).__name__}: {e}) → synthetic time courses")
        return _timing_synthetic_df()

tdf = load_timing_df(CONFIG)
print("timing df:", len(tdf), "rows |", tdf.subject.nunique(), "subjects | T =",
      len(tdf.attrs["times"]))

In [ ]:
# Task A5.1-4
def interaction_time_course(df, key, contrast_mode="proportion", contrasts=None):
    """Signed difference-of-differences per time bin for one process (elec-averaged).
    Return (times, effect). df['hg'] = per-trial time courses; df.attrs['times'] = time vector."""
    raise NotImplementedError("A5.1 — time-resolved interaction (d-o-d over time)")

def onset_50pct_peak(times, effect, expected_sign=+1):
    """First rising-flank crossing of 50% of the peak (amplitude-scale invariant)."""
    raise NotImplementedError("A5.2 — 50%-of-peak onset")

def peak_latency(times, effect, expected_sign=+1):
    """Time of the peak, reported alongside onset as a shape cross-check."""
    raise NotImplementedError("A5.3 — peak latency")

def jackknife_onset_difference(df_by_subject, expected_signs=(+1, +1), contrast_mode="proportion"):
    """Ulrich-Miller jackknifed LWPC-vs-LWPS onset difference over leave-one-out grand averages."""
    raise NotImplementedError("A5.4 — Ulrich-Miller jackknife of the onset difference")

#### Acceptance check A5
- **Latency–amplitude guard (unit test):** scaling a waveform by *k* leaves `onset_50pct_peak`
  unchanged — a pure amplitude difference cannot fake an onset difference.
- Onset is measured on **smooth leave-one-out grand averages** (the jackknife), never noisy
  per-subject single traces; report **onset and peak** together.

In [ ]:
t, e_stab = interaction_time_course(tdf, "stability")
_, e_flex = interaction_time_course(tdf, "flexibility")
on_s, pk_s = onset_50pct_peak(t, e_stab), peak_latency(t, e_stab)
on_f, pk_f = onset_50pct_peak(t, e_flex), peak_latency(t, e_flex)
print(f"LWPC (stability):  onset={on_s:.3f}s  peak={pk_s:.3f}s")
print(f"LWPS (flexibility): onset={on_f:.3f}s  peak={pk_f:.3f}s")

# latency-amplitude invariance unit test
k = 7.3
assert abs(onset_50pct_peak(t, e_stab) - onset_50pct_peak(t, k*e_stab)) < 1e-9, \
    "onset changed under pure amplitude scaling!"
print("✓ k-scaling unit test passed (onset invariant to amplitude)")

jk = jackknife_onset_difference(tdf)
print(f"\njackknife LWPC−LWPS onset difference = {jk['diff']:+.3f}s "
      f"(SE {jk['se']:.3f}, 95% CI {tuple(round(x,3) for x in jk['ci'])})")
print(f"Ulrich–Miller corrected t = {jk['t_corrected']:+.2f}, p = {jk['p']:.4g}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(t, e_stab, color="#2c7fb8", lw=2, label="LWPC (stability)")
ax.plot(t, e_flex, color="#d95f0e", lw=2, label="LWPS (flexibility)")
for on, pk, c in [(on_s, e_stab, "#2c7fb8"), (on_f, e_flex, "#d95f0e")]:
    ax.axvline(on, color=c, ls="--", lw=1.2)
    ax.axhline(0.5*np.nanmax(pk), color=c, ls=":", lw=.8, alpha=.6)
ax.axhline(0, color="k", lw=.6)
ax.set(title="Interaction time course + 50%-of-peak onset (dashed)",
       xlabel="time (s)", ylabel="interaction (difference-of-differences)")
ax.legend(); fig.tight_layout(); plt.show()

---
# A6 · Brain–behavior correlation

**Goal.** Tie the neural selectivity to the *actual* behavioral control adjustment, so the
substrates are shown to be functional. Two levels: **across subjects** (n = subjects, honest
but underpowered) and **within subject, single-trial** (preferred, high power), with a
**cross-pairing specificity control** (LWPC neural ↔ *switch* behavior should be weaker).

*Drop-in target:* new `stability_flexibility_brain_behavior.py`. Real behavioral magnitudes
come from `stats/erin_linear_mixed_effects_model.py` / `combinedData.csv` — **extracted**, not
recomputed. This sandbox fabricates a behavior table with a planted link so the functions are
testable anywhere.

*Hints:* `reveal("a6_hint1")`, `reveal("a6_hint2")`. *Answer:* `reveal("a6_solution")` / `load_reference("a6")`.

In [ ]:
# per-subject neural summary from A3's base_labels; fabricate a matching behavior table
_g = base_labels.groupby("subject")
_nS, _nF = _g["S"].sum(), _g["F"].sum()
_rng = np.random.default_rng(5)
behavior = pd.DataFrame({
    "subject": _nS.index,
    "beh_lwpc": 0.5*_nS.values + _rng.normal(0, 2, len(_nS)),    # planted link to n_S
    "beh_lwps": 0.5*_nF.values + _rng.normal(0, 2, len(_nF)),    # planted link to n_F
})

# fabricate single-trial rows: matched pairing carries a real slope, cross ~ 0
_rows = []
for s in sorted(base_labels.subject.unique()):
    n = 200
    hg_lwpc = _rng.normal(0, 1, n); hg_lwps = _rng.normal(0, 1, n)
    _rows.append(pd.DataFrame(dict(
        subject=str(s), hg_lwpc=hg_lwpc, hg_lwps=hg_lwps,
        cong_seq_adj=0.4*hg_lwpc + _rng.normal(0, 1, n),          # LWPC HG -> cong-seq RT adj
        switch_adj=0.02*hg_lwpc + 0.4*hg_lwps + _rng.normal(0, 1, n))))  # matched by hg_lwps
trial_df = pd.concat(_rows, ignore_index=True)
print("behavior table:", behavior.shape, "| trial_df:", trial_df.shape)

In [ ]:
# Task A6.1-2
def subject_level_brain_behavior(elec_labels, behavior):
    """Across-subject corr of neural selectivity (per-subject S/F counts) vs behavioral
    LWPC/LWPS magnitude. Return dict(corr_lwpc, p_lwpc, corr_lwps, p_lwps, n_subjects).
    Report the 'underpowered at n=subjects' caveat."""
    raise NotImplementedError("A6.1 — across-subject brain-behavior correlation")

def trialwise_brain_behavior(trial_df, group="LWPC"):
    """Within-subject single-trial mixed model (subject random effect): HG in a group ->
    the MATCHING RT adjustment. group in {'LWPC','LWPS'}. Return dict(slope, p, n_trials)."""
    raise NotImplementedError("A6.2 — within-subject single-trial mixed model")

#### Acceptance check A6
- Across-subject correlation reported **with its n** and the underpowered caveat.
- Within-subject model links the **matching** neural group to the matching behavioral
  adjustment, and the **cross** pairing (LWPC HG ↔ switch adjustment) is **weaker** — the
  specificity control.

In [ ]:
res = subject_level_brain_behavior(base_labels, behavior)
print(f"[across-subject]  corr(n_S, beh_LWPC) = {res['corr_lwpc']:+.2f} (p={res['p_lwpc']:.3f})")
print(f"                  corr(n_F, beh_LWPS) = {res['corr_lwps']:+.2f} (p={res['p_lwps']:.3f})")
print(f"                  n = {res['n_subjects']} subjects — {res['caveat']}\n")

matched = trialwise_brain_behavior(trial_df, group="LWPC")
# cross specificity control: same neural group vs the NON-matching behavior
_cross = smf.mixedlm("switch_adj ~ hg_lwpc", trial_df, groups=trial_df["subject"]).fit()
print(f"[within-subject]  matched  (LWPC HG → cong-seq adj): slope={matched['slope']:+.3f} "
      f"p={matched['p']:.2e}  (n={matched['n_trials']})")
print(f"[specificity]     cross    (LWPC HG → switch adj):   slope={_cross.params['hg_lwpc']:+.3f}")
print("→ matched slope should clearly exceed the cross-pairing slope")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
g = base_labels.groupby("subject")
m = pd.DataFrame({"subject": list(g.groups), "n_S": g["S"].sum().values,
                  "n_F": g["F"].sum().values}).merge(behavior, on="subject")
ax[0].scatter(m["n_S"], m["beh_lwpc"], color="#2c7fb8", s=40)
ax[0].set(title=f"Across-subject (LWPC): r={res['corr_lwpc']:+.2f}",
          xlabel="# stability electrodes", ylabel="behavioral LWPC magnitude")
ax[1].scatter(m["n_F"], m["beh_lwps"], color="#d95f0e", s=40)
ax[1].set(title=f"Across-subject (LWPS): r={res['corr_lwps']:+.2f}",
          xlabel="# flexibility electrodes", ylabel="behavioral LWPS magnitude")
fig.tight_layout(); plt.show()

---
## Where this code goes next

You now have working, tested versions of all the A1–A6 functions. To promote them:

1. **A1/A2** → `src/analysis/stats/stability_flexibility_segregation.py` (next to
   `per_electrode_labels` / `cmh_conjunction`).
2. **A3** → new `stability_flexibility_anatomy.py` (stats) + the `vis/` Glasser renderer for
   the brain plot. **A5** → new `stability_flexibility_timing.py`. **A6** → new
   `stability_flexibility_brain_behavior.py`. **A4** → new `decoding/cross_decoding.py`: the
   `_fit_predict` backend already wraps the project `Decoder` — pin `backend='decoder'` there
   and drop the `LogisticRegression` fallback.
3. Delete the corresponding `docs/skeletons/aN_*.py` stubs, and lift the acceptance checks
   here into `tests/`.

### The §0 cross-cutting principles, section by section
- **Disjoint halves / double-dipping** — A1 sign via `_interaction_effect` (disjoint-half
  pipeline); A4 disjoint train/test pseudo-trials.
- **Power matching** — the A2 threshold sweep (OR vs threshold, not one α).
- **FDR** across electrodes — A1 `per_electrode_anova_labels`.
- **Coverage bias** — A3 coverage-conditioned enrichment test.
- **Decoding confounds** — A4 per-condition mean removal + label-permutation null.
- **Latency–amplitude** — A5 50%-of-peak onset + peak latency + the k-scaling unit test.

### A4 · what's here vs. one remaining exercise
Implemented and runnable: pseudo-trials, per-condition mean removal, label-transfer
`cross_decode` (with the pluggable Decoder/​sklearn backend), the within-block baseline
(Fig 9), and temporal generalization (Fig 10). **Set-transfer (design b)** — train on one
electrode set, test on the other for the *same* label — reuses the exact same machinery
(swap the electrode subset instead of the contrast) and is left as a short exercise; see
`docs/skeletons/a4_cross_decoding.py`.